In [1]:
import warnings
from tqdm import TqdmWarning
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=TqdmWarning)
warnings.filterwarnings("ignore", message="Detected IPython.*")

import pandas as pd
from pathlib import Path
import re
import os
import json
import joblib
import numpy as np
import pyarrow.parquet as pq
import networkx as nx
from sklearn.model_selection import GridSearchCV
import utils as ut
import model_wrappers as mw
from hyperparameter import model_configs
from pathlib import Path


current_path = Path(__file__).resolve().parent if '__file__' in globals() else Path().resolve()



Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [2]:

base_path = current_path.parent / "data" / "silver" / "evaluation_benchmark"

# List all subdirectories in base_path
folders = [f for f in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, f))]

# Create a dictionary with folder name as key and full path as value
folder_dict = {f: base_path / f for f in folders}

save_path = current_path.parent / "data" / "gold" / "evaluation_benchmark"
save_path.mkdir(parents=True, exist_ok=True)

# Model training

In [3]:
import time

results_dfs = []

for folder_name, folder_path in folder_dict.items():
    print(f"\nProcessing dataset: {folder_name}")
    symbolic_equations = {"causal": {}, "traditional": {}}
    trained_models = {}
    best_params = {}
    results = []

    # Paths to experimental data
    train_path = folder_path / "train.csv"
    test_path = folder_path / "test.csv"
    intervention_path = folder_path / "interventions.json"
    adj_matrix_path = folder_path / "adj_matrix.csv"
    with open(intervention_path, "r") as f:
        interventions = json.load(f)["environments"]

    # Load train/test data without headers to determine column names
    temp_train_df = pd.read_csv(train_path, header=None)
    n_columns = temp_train_df.shape[1]
    column_names = [f"X{i}" for i in range(n_columns)]
    target_column = 'X' + str(interventions[0]['effect_idxs'][0]-1)

    # Reload with column names
    train_df = pd.read_csv(train_path, header=None, names=column_names)
    test_df = pd.read_csv(test_path, header=None, names=column_names)

    X_train = train_df.drop(columns=[target_column])
    y_train = train_df[target_column]
    X_test = test_df.drop(columns=[target_column])
    y_test = test_df[target_column]

    # Combine reference + test into one dataset
    combined_data = []
    for env in interventions:
        test_data = pd.DataFrame(env["test_data"], columns=column_names)
        ref_data = pd.DataFrame(env["reference_data"], columns=column_names)
        combined = pd.concat([test_data, ref_data], ignore_index=True)
        combined_data.append(combined)

    inter_df = pd.concat(combined_data)
    X_int = inter_df.drop(columns=[target_column])
    y_int = inter_df[target_column]

    # Load causal graph adjacency matrix
    adj_matrix = pd.read_csv(adj_matrix_path, header=None).values

    G = nx.DiGraph()
    for i, src in enumerate(column_names):
        for j, tgt in enumerate(column_names):
            if adj_matrix[i, j] == 1:
                G.add_edge(src, tgt)

    # Train and evaluate traditional and causal
    for name, cfg in model_configs.items():
        ModelClass = cfg["model_class"]
        params = cfg.get("params", {})
        param_grid = cfg.get("param_grid", None)

        # --- Causal version ---
        t0 = time.perf_counter()
        causal_models = ut.train_causal_models(train_df, G, ModelClass,
                                            model_params=params, param_grid=param_grid)
        causal_train_time = time.perf_counter() - t0

        causal_preds = ut.predict_causal(test_df, G, causal_models, what_if=False)
        rmse_c, wape_c, mae_c = ut.evaluate_metrics(test_df[target_column], causal_preds[target_column], normalize=True)
        results.append(["Causal", name, rmse_c, wape_c, mae_c, "Test", causal_train_time])

        if name == "SymbolicRegression":
            symbolic_equations["causal"][name] = {}
            for node, model in causal_models.items():
                try:
                    equation = str(model.model.get_best()['equation'])
                    equation = ut.round_numbers_in_string(equation)
                    symbolic_equations["causal"][name][node] = equation
                    print(f"[Causal {name}] Node '{node}': {equation}")
                except:
                    symbolic_equations["causal"][name][node] = "No equation available"

        # --- Traditional version ---
        if param_grid:
            base_model = ModelClass(**params)
            grid_search = GridSearchCV(base_model, param_grid=param_grid,
                                    cv=4, n_jobs=-1, scoring='neg_mean_absolute_error')
            t0 = time.perf_counter()
            grid_search.fit(X_train, y_train)
            trad_total_time = time.perf_counter() - t0
            best_params[name] = grid_search.best_params_
            print(f"[GridSearchCV] Best params for Traditional '{name}': {grid_search.best_params_}")

            # Refit with best params only (single fit)
            t0 = time.perf_counter()
            model = ModelClass(**{**params, **grid_search.best_params_})
            model.fit(X_train, y_train)
        else:
            t0 = time.perf_counter()
            model = ModelClass(**params)
            model.fit(X_train, y_train)
            trad_total_time = time.perf_counter() - t0  # No CV, same time
            best_params[name] = params

        y_pred = model.predict(X_test)
        rmse_t, wape_t, mae_t = ut.evaluate_metrics(y_test, y_pred, normalize=True)
        results.append(["Traditional", name, rmse_t, wape_t, mae_t, "Test", trad_total_time])

        if name == "SymbolicRegression":
            try:
                equation = str(model.model.get_best()['equation'])
                equation = ut.round_numbers_in_string(equation)
                symbolic_equations["traditional"][name] = equation
                print(f"[Traditional {name}] Target equation: {equation}")
            except:
                symbolic_equations["traditional"][name] = "No equation available"

        # Evaluate interventions
        causal_preds_int = ut.predict_causal(inter_df, G, causal_models, what_if=True)
        rmse_c_int, wape_c_int, mae_c_int = ut.evaluate_metrics(y_int, causal_preds_int[target_column], normalize=True)
        results.append(["Causal", name, rmse_c_int, wape_c_int, mae_c_int, "Intervention", causal_train_time])

        y_pred_int = model.predict(X_int)
        rmse_t_int, wape_t_int, mae_t_int = ut.evaluate_metrics(y_int, y_pred_int, normalize=True)
        results.append(["Traditional", name, rmse_t_int, wape_t_int, mae_t_int, "Intervention", trad_total_time])

        trained_models[name] = {"causal": causal_models, "traditional": model}

    # Results table
    results_df = pd.DataFrame(results, columns=["Model_Type", "Algorithm", "RMSE", "WAPE", "MAE", "Task", "train_time_total_s"])
    results_df = results_df[["Model_Type", "Algorithm", "MAE", "RMSE", "WAPE", "Task", "train_time_total_s"]]
    results_df['experiment'] = folder_name

    results_dfs.append(results_df)

    # Display symbolic equations
    print("\n" + "="*80)
    print("SYMBOLIC REGRESSION EQUATIONS")
    print("="*80)

    if symbolic_equations["causal"]:
        print("\n--- CAUSAL MODELS ---")
        for model_name, node_equations in symbolic_equations["causal"].items():
            print(f"\n{model_name}:")
            for node, equation in node_equations.items():
                print(f"  {node}: {equation}")

    if symbolic_equations["traditional"]:
        print("\n--- TRADITIONAL MODELS ---")
        for model_name, equation in symbolic_equations["traditional"].items():
            print(f"{model_name}: {equation}")

    print("\n" + "="*80)
    print(results_df)

    # --- Save models and best hyperparameters per dataset ---
    folder_save_path = save_path / folder_name
    models_dir = folder_save_path / "models"
    models_dir.mkdir(parents=True, exist_ok=True)

    # Save raw results as parquet
    results_df.to_parquet(folder_save_path / "results.parquet", index=False, compression="snappy")

    # Save per-dataset filtered tables
    fmt_df = results_df.copy()
    fmt_df['Model_Type'] = fmt_df['Model_Type'].replace({'Causal': 'CML', 'Traditional': 'ML'})
    fmt_df['Algorithm'] = fmt_df['Algorithm'].replace({'RandomForest': 'RForest', 'SymbolicRegression': 'Symbolic',
                                                        'XGBRegression': 'XGB', 'LinearRegression': 'Linreg'})
    ind_intervention = fmt_df[fmt_df["Task"] == "Intervention"].drop(columns=["Task", "experiment"]).sort_values("MAE")
    ind_test = fmt_df[fmt_df["Task"] == "Test"].drop(columns=["Task", "experiment"]).sort_values("MAE")
    ind_intervention.to_parquet(folder_save_path / "intervention_results.parquet", index=False, compression="snappy")
    ind_test.to_parquet(folder_save_path / "test_results.parquet", index=False, compression="snappy")


    # Save best hyperparameters as JSON
    params_path = folder_save_path / "best_hyperparameters.json"
    serialisable_params = {
        model_name: {k: (list(v) if isinstance(v, tuple) else v) for k, v in p.items()}
        for model_name, p in best_params.items()
    }
    with open(params_path, "w") as f:
        json.dump(serialisable_params, f, indent=2)


    # Save trained models
    for model_name, model_dict in trained_models.items():
        joblib.dump(model_dict["traditional"], models_dir / f"{model_name}_traditional.pkl")
        for node, causal_model in model_dict["causal"].items():
            joblib.dump(causal_model, models_dir / f"{model_name}_causal_{node}.pkl")




Processing dataset: csuite_nonlin_simpson
[GridSearchCV] Best params for Traditional 'RandomForest': {'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 15, 'n_estimators': 500}
[GridSearchCV] Best params for Traditional 'LinearRegression': {'fit_intercept': True, 'positive': False}
[GridSearchCV] Best params for Traditional 'MLP': {'alpha': 0.1, 'hidden_layer_sizes': (50, 50), 'learning_rate': 'constant'}
[GridSearchCV] Best params for Traditional 'XGBoost': {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 1000, 'subsample': 0.7}
[GridSearchCV] Best params for Traditional 'GAM': {'lam': 1, 'n_splines': 20, 'spline_order': 4}
Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
Detect

In [4]:
all_results_df = pd.concat(results_dfs, ignore_index=True)

# Group by Model_Type, Algorithm, and Task, and take the median of the numeric columns
aggregated_results_df = all_results_df.groupby(['Model_Type', 'Algorithm', 'Task']).agg({
    'MAE': 'median',
    'RMSE': 'median',
    'WAPE': 'median',
    'train_time_total_s': 'median',
}).reset_index()


aggregated_results_df


,Model_Type,Algorithm,Task,MAE,RMSE,WAPE,train_time_total_s
0,Causal,GAM,Intervention,1.046519,1.368973,104.651947,1.467259
1,Causal,GAM,Test,0.194971,0.243392,24.991562,1.467259
2,Causal,LinearRegression,Intervention,1.084992,1.318023,108.499153,0.200048
3,Causal,LinearRegression,Test,0.284212,0.364265,35.129994,0.200048
4,Causal,MLP,Intervention,1.040650,1.363747,104.064962,49.339763
5,Causal,MLP,Test,0.195665,0.244455,25.080096,49.339763
6,Causal,RandomForest,Intervention,1.048175,1.386996,104.817543,653.489249
7,Causal,RandomForest,Test,0.197230,0.246197,25.279935,653.489249
8,Causal,SymbolicRegression,Intervention,1.085327,1.352616,108.532698,646.131138
9,Causal,SymbolicRegression,Test,0.210935,0.267074,27.067391,646.131138


In [5]:
df = aggregated_results_df.copy()

columns_to_display = ['Type', 'Model', 'MAE', 'RMSE', 'WAPE']

df['Model_Type'] = df['Model_Type'].replace({'Causal': 'CML', 'Traditional': 'ML'})
df['Algorithm'] = df['Algorithm'].replace({'RandomForest': 'RForest', 'SymbolicRegression': 'Symbolic',
                                           'XGBRegression': 'XGB', 'LinearRegression': 'Linreg',})

df.columns = ["Type", "Model", "Task", "MAE", "RMSE", "WAPE", "train_time_total_s"]

# Filter for Intervention task and drop Task column
intervention_df = df[df["Task"] == "Intervention"].drop(columns=["Task"]).sort_values(by=["MAE"]).copy()

# Filter for Test task and drop Task column
test_df = df[df["Task"] == "Test"].drop(columns=["Task"]).sort_values(by=["MAE"]).copy()

for col in ["MAE", "RMSE", "WAPE", "train_time_total_s"]:
    intervention_df[col] = pd.to_numeric(intervention_df[col], errors="coerce")
    test_df[col] = pd.to_numeric(test_df[col], errors="coerce")

fmt = "{:,.4f}"
fmt_t = "{:,.4f}s"

print("Intervention Results:")

display(intervention_df[columns_to_display].style.format({"MAE": fmt, "RMSE": fmt, "WAPE": fmt,
                                      "train_time_total_s": fmt_t}).hide(axis="index"))

print("\nTest Results:")
display(test_df[columns_to_display].style.format({"MAE": fmt, "RMSE": fmt, "WAPE": fmt,
                               "train_time_total_s": fmt_t}).hide(axis="index"))

display(intervention_df.style.format({"MAE": fmt, "RMSE": fmt, "WAPE": fmt,
                                      "train_time_total_s": fmt_t}).hide(axis="index"))
display(test_df.style.format({"MAE": fmt, "RMSE": fmt, "WAPE": fmt,
                               "train_time_total_s": fmt_t}).hide(axis="index"))

# Save aggregated tables
intervention_df.to_parquet(save_path / "aggregated_intervention_results.parquet", index=False, compression="snappy")
test_df.to_parquet(save_path / "aggregated_test_results.parquet", index=False, compression="snappy")


Intervention Results:


Type,Model,MAE,RMSE,WAPE
CML,MLP,1.0406,1.3637,104.0650
CML,XGBoost,1.0407,1.3721,104.0680
CML,GAM,1.0465,1.3690,104.6519
CML,RForest,1.0482,1.3870,104.8175
CML,Linreg,1.0850,1.3180,108.4992
CML,Symbolic,1.0853,1.3526,108.5327
ML,XGBoost,1.1224,1.4292,112.2403
ML,GAM,1.1231,1.4337,112.3113
ML,MLP,1.1275,1.4411,112.7521
ML,RForest,1.1403,1.4528,114.0344



Test Results:


Type,Model,MAE,RMSE,WAPE
ML,GAM,0.1898,0.2374,24.3297
ML,MLP,0.1900,0.2387,24.3636
ML,XGBoost,0.1907,0.2400,24.4475
ML,RForest,0.1928,0.2424,24.7198
CML,GAM,0.1950,0.2434,24.9916
CML,MLP,0.1957,0.2445,25.0801
CML,RForest,0.1972,0.2462,25.2799
CML,XGBoost,0.1980,0.2475,25.3799
CML,Symbolic,0.2109,0.2671,27.0674
ML,Symbolic,0.2145,0.2722,27.5077


Type,Model,MAE,RMSE,WAPE,train_time_total_s
CML,MLP,1.0406,1.3637,104.0650,49.3398s
CML,XGBoost,1.0407,1.3721,104.0680,150.2566s
CML,GAM,1.0465,1.3690,104.6519,1.4673s
CML,RForest,1.0482,1.3870,104.8175,653.4892s
CML,Linreg,1.0850,1.3180,108.4992,0.2000s
CML,Symbolic,1.0853,1.3526,108.5327,646.1311s
ML,XGBoost,1.1224,1.4292,112.2403,102.6645s
ML,GAM,1.1231,1.4337,112.3113,0.9880s
ML,MLP,1.1275,1.4411,112.7521,8.2433s
ML,RForest,1.1403,1.4528,114.0344,255.0984s


Type,Model,MAE,RMSE,WAPE,train_time_total_s
ML,GAM,0.1898,0.2374,24.3297,0.9880s
ML,MLP,0.1900,0.2387,24.3636,8.2433s
ML,XGBoost,0.1907,0.2400,24.4475,102.6645s
ML,RForest,0.1928,0.2424,24.7198,255.0984s
CML,GAM,0.1950,0.2434,24.9916,1.4673s
CML,MLP,0.1957,0.2445,25.0801,49.3398s
CML,RForest,0.1972,0.2462,25.2799,653.4892s
CML,XGBoost,0.1980,0.2475,25.3799,150.2566s
CML,Symbolic,0.2109,0.2671,27.0674,646.1311s
ML,Symbolic,0.2145,0.2722,27.5077,124.8927s


# Individual results for the datasets

In [6]:
all_results_df = pd.concat(results_dfs, ignore_index=True)

for experiment in all_results_df['experiment'].unique():

    print(experiment)
    df = all_results_df[all_results_df['experiment'] == experiment].copy()

    df['Model_Type'] = df['Model_Type'].replace({'Causal': 'CML', 'Traditional': 'ML'})
    df['Algorithm'] = df['Algorithm'].replace({'RandomForest': 'RForest', 'SymbolicRegression': 'Symbolic',
                                            'XGBRegression': 'XGB', 'LinearRegression': 'Linreg',})

    df.columns = ["Type", "Model", "MAE", "RMSE", "WAPE", "Task", "train_time_total_s", "experiment"]

    # Filter for Intervention task and drop Task column
    intervention_df = df[df["Task"] == "Intervention"].drop(columns=["Task", "experiment"]).sort_values(by=["MAE"]).copy()
    test_df = df[df["Task"] == "Test"].drop(columns=["Task", "experiment"]).sort_values(by=["MAE"]).copy()

    for col in ["MAE", "RMSE", "WAPE"]:
        intervention_df[col] = pd.to_numeric(intervention_df[col], errors="coerce")
        test_df[col] = pd.to_numeric(test_df[col], errors="coerce")

    fmt = "{:,.4f}"

    print("Intervention Results:")
    display(intervention_df.style.format({"MAE": fmt, "RMSE": fmt, "WAPE": fmt}).hide(axis="index"))

    print("\nTest Results:")
    display(test_df.style.format({"MAE": fmt, "RMSE": fmt, "WAPE": fmt}).hide(axis="index"))


csuite_nonlin_simpson
Intervention Results:


Type,Model,MAE,RMSE,WAPE,train_time_total_s
ML,MLP,0.8433,1.0505,84.3254,3.185640
ML,XGBoost,0.8674,1.0851,86.7379,54.224364
ML,RForest,0.8743,1.0910,87.4261,164.316502
ML,Linreg,0.8825,1.0662,88.2500,0.039127
ML,GAM,0.8983,1.1190,89.8329,0.523204
CML,XGBoost,0.9179,1.1459,91.7855,88.095770
CML,RForest,0.9185,1.1469,91.8538,461.491490
CML,MLP,0.9193,1.1493,91.9269,38.638544
CML,GAM,0.9196,1.1470,91.9647,0.857732
CML,Symbolic,0.9221,1.1442,92.2057,354.230524



Test Results:


Type,Model,MAE,RMSE,WAPE,train_time_total_s
ML,MLP,0.1601,0.2026,20.0289,3.185640
ML,GAM,0.1629,0.2062,20.3795,0.523204
ML,XGBoost,0.1630,0.2075,20.3865,54.224364
ML,RForest,0.1656,0.2108,20.7160,164.316502
CML,GAM,0.1690,0.2121,21.1440,0.857732
CML,MLP,0.1698,0.2132,21.2388,38.638544
CML,RForest,0.1714,0.2151,21.4415,461.491490
CML,Symbolic,0.1721,0.2155,21.5220,354.230524
CML,XGBoost,0.1728,0.2168,21.6197,88.095770
ML,Symbolic,0.1791,0.2258,22.4029,75.491665


csuite_large_backdoor
Intervention Results:


...

Type,Model,MAE,RMSE,WAPE,train_time_total_s
ML,GAM,0.2166,0.2686,28.2799,1.481776
ML,XGBoost,0.2184,0.2724,28.5085,159.363093
ML,MLP,0.2198,0.2748,28.6983,10.056989
ML,RForest,0.2200,0.2739,28.7236,341.702019
CML,GAM,0.2209,0.2747,28.8391,3.198146
CML,MLP,0.2215,0.2757,28.9214,72.732535
CML,RForest,0.2230,0.2773,29.1184,1066.908681
CML,XGBoost,0.2232,0.2782,29.1401,313.983043
ML,Linreg,0.2380,0.3002,31.0756,0.040755
CML,Linreg,0.2495,0.3174,32.5686,0.283574


csuite_symprod_simpson
Intervention Results:


Type,Model,MAE,RMSE,WAPE,train_time_total_s
ML,XGBoost,0.9896,1.3124,98.9574,52.446516
ML,RForest,0.9935,1.3193,99.3538,168.494815
CML,XGBoost,0.9990,1.3263,99.8992,101.985650
CML,RForest,0.9992,1.3304,99.9221,335.798324
CML,GAM,0.9996,1.3305,99.9632,0.856130
ML,GAM,0.9999,1.3296,99.9869,0.520335
CML,MLP,0.9999,1.3294,99.9927,41.264229
ML,Symbolic,1.0080,1.3342,100.7989,243.586400
CML,Symbolic,1.0080,1.3342,100.7989,464.761947
ML,MLP,1.0090,1.3422,100.8951,6.922607



Test Results:


Type,Model,MAE,RMSE,WAPE,train_time_total_s
CML,GAM,0.0705,0.1030,7.6111,0.856130
CML,MLP,0.0710,0.1033,7.6670,41.264229
ML,GAM,0.0712,0.1041,7.6874,0.520335
CML,XGBoost,0.0725,0.1046,7.8297,101.985650
ML,XGBoost,0.0726,0.1042,7.8395,52.446516
ML,MLP,0.0727,0.1050,7.8493,6.922607
CML,RForest,0.0729,0.1051,7.8763,335.798324
ML,RForest,0.0732,0.1047,7.9014,168.494815
ML,Symbolic,0.0884,0.1207,9.5415,243.586400
CML,Symbolic,0.0884,0.1207,9.5415,464.761947
